## Run GraphST

In [1]:
from numpy.random import default_rng
import scanpy as sc
# import squidpy as sq
from anndata import AnnData
import scipy
sc.logging.print_header()
from sklearn.metrics.cluster import adjusted_rand_score
import numpy as np
import pandas as pd
import seaborn as sns
import os
import torch
import pandas as pd
from sklearn import metrics
import multiprocessing as mp
from sklearn.metrics.cluster import normalized_mutual_info_score
from GraphST import GraphST

from GraphST import GraphST
from sklearn.decomposition import PCA
from GraphST.utils import refine_label
from sklearn.preprocessing import StandardScaler
from GraphST.utils import mclust_R

/home/vanhoan310/miniconda3/envs/gnn/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


scanpy==1.9.3 anndata==0.9.2 umap==0.5.7 numpy==1.24.4 scipy==1.8.1 pandas==1.4.2 scikit-learn==1.3.2 statsmodels==0.14.1 python-igraph==0.11.8 pynndescent==0.5.13


In [2]:
adata = sc.read_visium('/data/hoan/spatial_transcriptomics/GraphBG-analysis/Data', count_file='filtered_feature_bc_matrix.h5')
ground_truth = pd.read_csv('/data/hoan/spatial_transcriptomics/GraphBG-analysis/Data/metadata.tsv', delimiter='\t', index_col=0)
adata.obs['ground_truth'] = ground_truth['ground_truth']
adata.var_names_make_unique()

/home/vanhoan310/miniconda3/envs/gnn/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [3]:
# Run device, by default, the package is implemented on 'cpu'. We recommend using GPU.
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
os.environ['R_HOME'] = '/data/hoan/mybin/miniconda3/lib/R'

true_labels = pd.factorize(adata.obs['ground_truth'])[0] 

n_clusters = len(np.unique(true_labels))

model = GraphST.GraphST(adata, device=device)
# train model
adata = model.train()

if scipy.sparse.issparse(adata.X):
    adata.obsm['raw'] = adata.X.todense() # this is already good
else:
    adata.obsm['raw'] = adata.X
adata.obsm["raw"] = np.asarray(adata.obsm["raw"])

from sklearn.decomposition import PCA
from GraphST.utils import refine_label
from sklearn.preprocessing import StandardScaler
from GraphST.utils import mclust_R

# Run GraphST
from GraphST.utils import clustering
clustering(adata, n_clusters=n_clusters, radius=50, key='emb', refinement=True)
adata.obs['GraphST'] = adata.obs['domain']

Begin to train ST data...


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 600/600 [01:44<00:00,  5.74it/s]


Optimization finished for ST data!


During startup - Warning messages:
1: package ‘methods’ was built under R version 3.6.1 
2: package ‘datasets’ was built under R version 3.6.1 
3: package ‘utils’ was built under R version 3.6.1 
4: package ‘grDevices’ was built under R version 3.6.1 
5: package ‘graphics’ was built under R version 3.6.1 
6: package ‘stats’ was built under R version 3.6.1 
R[write to console]:     __  ___________    __  _____________
   /  |/  / ____/ /   / / / / ___/_  __/
  / /|_/ / /   / /   / / / /\__ \ / /   
 / /  / / /___/ /___/ /_/ /___/ // /    
/_/  /_/\____/_____/\____//____//_/    version 5.4.7
Type 'citation("mclust")' for citing this R package in publications.



fitting ...
  |======================================================================| 100%


In [4]:
NMI = normalized_mutual_info_score(adata.obs['GraphST'], adata.obs['ground_truth'])
print(NMI)

0.6696001645606913


In [5]:
adata.write_h5ad(
    '/data/hoan/spatial_transcriptomics/GraphBG-analysis/Data/Section_1_GraphST.h5ad'
)